# 00. Why randomize?

Before using the library, it helps to understand the problem randomization solves. This notebook shows, with a small simulation, why comparing treated and control units **without** randomizing leads to wrong conclusions.

## The fundamental problem

Each unit has two potential outcomes, `Y(0)` and `Y(1)`, but we observe only one. We cannot measure the effect on a single unit. The way out is to estimate the **average** effect by comparing groups, and that is where the risk appears: if those who receive the treatment are systematically different from those who do not, the difference in means mixes the effect with that difference.

## When treatment choice is biased

We simulate a confounder `x` that affects both the outcome and the chance the unit chooses the treatment (self-selection). The true effect is `1.0`.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
n = 2000

x = rng.normal(size=n)                 # confounder: affects treatment and outcome
tau = 1.0                              # true effect
y0 = 2.0 * x + rng.normal(size=n)      # Y(0) depends on x
y1 = y0 + tau                          # Y(1)

# Self-selection: units with high x tend to choose the treatment
p_self = 1.0 / (1.0 + np.exp(-x))
t_self = (rng.uniform(size=n) < p_self).astype(int)
y_self = np.where(t_self == 1, y1, y0)

naive = y_self[t_self == 1].mean() - y_self[t_self == 0].mean()
print(f"Naive estimate (confounded): {naive:.3f}  | true: {tau}")

The naive estimate lands **well above** `1.0`: the treated have higher `x`, hence higher baseline `Y(0)`. The difference in means credited the treatment with something that belonged to the confounder.

## Randomization breaks confounding

Now we assign treatment **at random** with `CRD`, ignoring `x`. Because the assignment is independent of `x`, the two groups are comparable and the difference in means estimates the true effect.

In [ ]:
from skxperiments.core.assignment import CRDAssignment
from skxperiments.design.crd import CRD
from skxperiments.estimators.difference_in_means import DifferenceInMeans

df = pd.DataFrame({"x": x})
design = CRD(p=0.5, seed=7)
assignment = design.randomize(df)

t = assignment.data_[assignment.treatment_col_].to_numpy()
data = assignment.data_.copy()
data["y"] = np.where(t == 1, y1, y0)
assignment = CRDAssignment(
    data=data, treatment_col=assignment.treatment_col_, design=design, seed=7
)

ate = DifferenceInMeans(outcome_col="y").fit(assignment).estimate().ate
print(f"Randomized estimate: {ate:.3f}  | true: {tau}")

## What we learned

- Without randomization, the difference in means conflates the effect with pre-existing differences between the groups.
- Randomizing makes the groups comparable **by construction**, with no need to measure or model the confounder.
- This is why the library starts from the **design**: `CRD` (and the other designs) is what guarantees this comparability.

**Next:** `01. Your first experiment` shows the full end-to-end flow.